<img src="logos/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg.png"
     width="250"
     style="display: block; margin-left: auto; margin-right: auto;">


# Titanic – Data Preparation & Feature Engineering (Industrial Standard)

This notebook builds a reproducible preprocessing + feature engineering pipeline.
Key principles:
- Split first (to avoid leakage).
- Fit preprocessing only on train.
- Use sklearn Pipeline / ColumnTransformer.
- Output: `X_train`, `X_test`, `y_train`, `y_test`, and `preprocess_pipeline`.

No modeling is performed here.

# Imports

In [7]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import re

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer

warnings.filterwarnings("ignore")

# setting the seed
RANDOM_STATE = 42
TEST_SIZE = 0.2
DATA_DIR = Path("../data")
TARGET_COL = "Survived"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Load Data

In [4]:
titanic_path = f"{DATA_DIR}/titanic_data.csv"
df = pd.read_csv(titanic_path)

print("Shape:", df.shape)
df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Target + split early

In this section we will split the dataset in two parts:

- Train
- Test

And select the target column that will be our Y. In this case as we decided in our EDA, our target will be "Survived"

In [6]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train survival rate:", float(y_train.mean()))
print("Test survival rate:", float(y_test.mean()))

Train shape: (712, 11) Test shape: (179, 11)
Train survival rate: 0.38342696629213485
Test survival rate: 0.3854748603351955


# Feature engineering functions

In [8]:
def extract_title(name: str) -> str:
    """
    Extract a title from a Titanic passenger name.
    Examples: Mr, Mrs, Miss, Master. Returns 'Other' for rare titles.
    """
    if not isinstance(name, str) or not name:
        return "Unknown"

    match = re.search(r",\s*([^\.]+)\.", name)
    if not match:
        return "Unknown"

    title = match.group(1).strip()

    common_titles = {"Mr", "Mrs", "Miss", "Master"}
    return title if title in common_titles else "Other"


def extract_deck(cabin: str) -> str:
    """
    Extract the deck letter from cabin, or 'Missing' if not available.
    """
    if not isinstance(cabin, str) or not cabin:
        return "Missing"
    return cabin[0].upper()


def add_engineered_features(X_in: pd.DataFrame) -> pd.DataFrame:
    """
    Create engineered Titanic features:
    - title from Name
    - deck from Cabin
    - has_cabin flag
    - family_size and is_alone
    """
    X = X_in.copy()

    X["title"] = X["Name"].apply(extract_title)
    X["deck"] = X["Cabin"].apply(extract_deck)
    X["has_cabin"] = X["Cabin"].notna().astype(int)

    X["family_size"] = X["SibSp"].fillna(0).astype(int) + X["Parch"].fillna(0).astype(int) + 1
    X["is_alone"] = (X["family_size"] == 1).astype(int)

    return X

**Applying the functions**

In [9]:
X_train_fe = add_engineered_features(X_train)
X_test_fe = add_engineered_features(X_test)

X_train_fe.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,title,deck,has_cabin,family_size,is_alone
692,693,3,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,NaN,S,Mr,Missing,0,1,1
481,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,NaN,S,Mr,Missing,0,1,1
527,528,1,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,Mr,C,1,1,1
855,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,NaN,S,Mrs,Missing,0,2,0
801,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,NaN,S,Mrs,Missing,0,3,0


# Feature Selection

Step 1: features that must be dropped for the two datasets (train and test):

- PassengerId
- Name (We extracted the title from here, but the name does not add information)
- Cabin (replaced by `deck` and `has_cabin`)
- Ticket

In [10]:
drop_cols = [
    "PassengerId",  # identifier
    "Name",         # replaced by title
    "Cabin",        # replaced by deck/has_cabin
    "Ticket",       # high cardinality (initially drop)
]

X_train_fe = X_train_fe.drop(columns=drop_cols, errors="ignore")
X_test_fe = X_test_fe.drop(columns=drop_cols, errors="ignore")

print("After FE + drop:", X_train_fe.shape, X_test_fe.shape)

After FE + drop: (712, 12) (179, 12)


**Column typing:** in this next cell we will group our features by types creating lists for each type

In [11]:
numeric_features = ["Age", "Fare", "SibSp", "Parch", "family_size"]
ordinal_features = ["Pclass"]
categorical_features = ["Sex", "Embarked", "title", "deck"]
binary_features = ["has_cabin", "is_alone"]

# Ensure only existing columns are used (robustness)
def keep_existing(cols: list[str], df_: pd.DataFrame) -> list[str]:
    return [c for c in cols if c in df_.columns]

numeric_features = keep_existing(numeric_features, X_train_fe)
ordinal_features = keep_existing(ordinal_features, X_train_fe)
categorical_features = keep_existing(categorical_features, X_train_fe)
binary_features = keep_existing(binary_features, X_train_fe)

numeric_features, ordinal_features, categorical_features, binary_features

(['Age', 'Fare', 'SibSp', 'Parch', 'family_size'],
 ['Pclass'],
 ['Sex', 'Embarked', 'title', 'deck'],
 ['has_cabin', 'is_alone'])

## Log Transformation of Fare

This cell applies a log(1 + Fare) transformation to reduce right-skewness.
Fare is highly skewed, and log transformation stabilizes variance and
improves linear model behavior.

In [13]:
if "Fare" in X_train_fe.columns:
    X_train_fe["Fare"] = np.log1p(X_train_fe["Fare"])
    X_test_fe["Fare"] = np.log1p(X_test_fe["Fare"])

## Preprocessing Pipeline Architecture

In this cell we define a `ColumnTransformer`, which allows us to apply
different preprocessing steps to different groups of features in a
structured and reproducible way.

### Why is this important?

Real-world datasets contain heterogeneous data types:

- Numerical features (e.g., Age, Fare)
- Ordinal features (e.g., Pclass)
- Nominal categorical features (e.g., Sex, Embarked)
- Binary engineered features (e.g., is_alone)

Each type requires different preprocessing strategies.
Applying the same transformation to all columns would be incorrect.

---

### What does ColumnTransformer do?

`ColumnTransformer` allows us to:

- Apply **median imputation** to numeric features.
- Apply **mode imputation + one-hot encoding** to categorical features.
- Keep ordinal variables in numeric form.
- Pass binary features unchanged.

All transformations are grouped into a single object called
`preprocess_pipeline`.

---

### Why use a Pipeline instead of manual transformations?

1. **Prevents data leakage**  
   The preprocessing will be fitted only on training data in the modeling stage.

2. **Ensures reproducibility**  
   The exact same transformations will be applied to train and test data.

3. **Industrial standard**  
   Modern ML systems rely on pipelines for maintainability and deployment.

---

### Key Concept

The preprocessing pipeline becomes part of the final model.

In production, we do not only deploy the model —
we deploy the preprocessing + model together as a single object.

In [14]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

ordinal_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # keep as numeric 1/2/3 (already ordinal). No encoding needed.
    ]
)

preprocess_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("ord", ordinal_transformer, ordinal_features),
        ("cat", categorical_transformer, categorical_features),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop",
)

## Final Feature Matrices

This cell prepares the final training and test feature matrices
after feature engineering and column selection.

These datasets are the inputs for modeling in the next notebook.
No fitting is performed at this stage.

In [15]:
# These are the objects that the next Notebook will import/use.
X_train_final = X_train_fe.copy()
X_test_final = X_test_fe.copy()

print("Final train features shape:", X_train_final.shape)
print("Final test features shape:", X_test_final.shape)

Final train features shape: (712, 12)
Final test features shape: (179, 12)


## Artifact Persistence

This cell saves:
- The preprocessing pipeline.
- The train/test split datasets.

Persisting artifacts ensures reproducibility and enables
seamless integration with the modeling notebook.

In [16]:
import joblib

artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocess_pipeline, artifacts_dir / "preprocess_pipeline.joblib")
joblib.dump((X_train_final, X_test_final, y_train, y_test), artifacts_dir / "split_data.joblib")

['../artifacts/split_data.joblib']

# Output Summary

This notebook produces:
- `X_train_final`, `X_test_final`, `y_train`, `y_test`
- `preprocess_pipeline` (ColumnTransformer)

Key design choices:
- Split performed before any transformations to avoid leakage.
- High-cardinality / raw text fields dropped (Ticket, Name), replaced with engineered features.
- Cabin replaced by `deck` + `has_cabin`.
- Family features created (`family_size`, `is_alone`).
- Numeric and categorical preprocessing handled via sklearn pipelines.

Next notebook: Baseline modeling, model comparison, Optuna tuning, interpretation, and error analysis.